# BiLSTM Model Performance Evaluation
**Sensor EM-300 — 24h Temperature & Humidity Forecasting**

---

| Section | หัวข้อ |
|---|---|
| 0 | Setup & Load |
| 1 | Training History |
| 2 | Overall Metrics (Train / Val / Test) |
| 3 | Per-Horizon Metrics (h+1 → h+24) |
| 4 | Actual vs Predicted — Time Series |
| 5 | Scatter & Residuals |
| 6 | Best / Worst Prediction Windows |
| 7 | Error Heatmap (Hour of Day) |
| 8 | Summary Dashboard |


## 0. Setup & Load

In [ ]:
import sys, os, warnings, pickle
from pathlib import Path

# ── project root ─────────────────────────────────────────────────────────
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 150,
                     'axes.spines.top': False, 'axes.spines.right': False})
%matplotlib inline

# ── paths ─────────────────────────────────────────────────────────────────
MODEL_DIR  = Path('models')
DATA_DIR   = Path('data/processed')
FIG_DIR    = Path('notebooks')

# ── imports from trainer ──────────────────────────────────────────────────
from src.trainnig.lstm_trainer import (
    add_time_features, make_sequences, compute_metrics,
    load_lstm_bundle, NumpyScaler,
    LOOKBACK_STEPS, N_HOURS_AHEAD, STEPS_PER_HOUR, N_FEATURES, FEATURE_NAMES,
)
print('Imports OK')
print(f'Lookback : {LOOKBACK_STEPS} steps = {LOOKBACK_STEPS*10//60}h')
print(f'Forecast : {N_HOURS_AHEAD}h ahead (hourly)')


In [ ]:
# ── Load splits ──────────────────────────────────────────────────────────
def load_split(name):
    df = pd.read_csv(DATA_DIR / name, index_col=0, parse_dates=True)
    if df.index.tzinfo is not None:
        df.index = df.index.tz_convert('Asia/Bangkok').tz_localize(None)
    return df[['temp','hum']].dropna()

train_df = load_split('train.csv')
val_df   = load_split('val.csv')
test_df  = load_split('test.csv')

n_total = len(train_df) + len(val_df) + len(test_df)
print(f'Train : {len(train_df):>5,}  ({len(train_df)/n_total*100:.0f}%)')
print(f'Val   : {len(val_df):>5,}  ({len(val_df)/n_total*100:.0f}%)')
print(f'Test  : {len(test_df):>5,}  ({len(test_df)/n_total*100:.0f}%)')
print(f'\nTest range: {test_df.index.min()} → {test_df.index.max()}')


In [ ]:
# ── Helper: evaluate full split → y_true (N,24), y_pred (N,24) ──────────
def evaluate_split(df, target):
    col_idx = 0 if target == 'temp' else 1
    bundle  = load_lstm_bundle(target, MODEL_DIR)
    model   = bundle['model']
    sx      = bundle['scaler_X']
    sy      = bundle['scaler_y']

    raw   = add_time_features(df)        # (N, 8)
    Xs    = sx.transform(raw)            # standardized
    X, ys = make_sequences(Xs, col_idx) # (N_seq, 144, 8), (N_seq, 24)

    y_pred_s = model.predict(X, batch_size=256, verbose=0)  # (N_seq, 24)
    # inverse scale each horizon independently
    y_pred = np.zeros_like(y_pred_s)
    y_true = np.zeros_like(ys)
    for h in range(N_HOURS_AHEAD):
        y_pred[:, h] = sy.inverse_transform(y_pred_s[:, h].reshape(-1,1)).flatten()
        y_true[:, h] = sy.inverse_transform(ys[:, h].reshape(-1,1)).flatten()
    return y_true, y_pred

print('Helper ready — run evaluation cells below')


---
## 1. Training History

Loss curves ระหว่าง training และ training summary จาก `lstm_training_summary.csv`

In [ ]:
# ── Training Summary ─────────────────────────────────────────────────────
summary = pd.read_csv(MODEL_DIR / 'lstm_training_summary.csv')
display(summary.set_index('target').round(4))


In [ ]:
# ── Loss curve images (saved during training) ─────────────────────────────
hist_files = [
    ('lstm_history_temp.png', 'Temperature — Training Loss'),
    ('lstm_history_hum.png',  'Humidity — Training Loss'),
]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (fname, title) in zip(axes, hist_files):
    p = MODEL_DIR / fname
    if p.exists():
        img = plt.imread(str(p))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=11)
    else:
        ax.text(0.5, 0.5, f'{fname} not found', ha='center', va='center',
                transform=ax.transAxes, color='red')
        ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig1_history.png', bbox_inches='tight')
plt.show()


---
## 2. Overall Metrics — Train / Val / Test

เปรียบเทียบ RMSE, MAE, MAPE, R² ข้ามทุก split เพื่อตรวจ overfit

In [ ]:
# ── Evaluate ทุก split ───────────────────────────────────────────────────
# อาจใช้เวลา 1-2 นาที (TF loading + inference)
results = {}
for target in ['temp', 'hum']:
    results[target] = {}
    for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        yt, yp = evaluate_split(df, target)
        m = compute_metrics(yt.flatten(), yp.flatten())
        results[target][split_name] = {'y_true': yt, 'y_pred': yp, 'metrics': m}
        print(f'{target:>4} | {split_name:<5} | '
              f'RMSE={m["RMSE"]:.3f}  MAE={m["MAE"]:.3f}  '
              f'MAPE={m["MAPE"]:.2f}%  R2={m["R2"]:.4f}')
print('\nDone')


In [ ]:
# ── Metrics table ────────────────────────────────────────────────────────
rows = []
for target in ['temp', 'hum']:
    unit = 'degC' if target == 'temp' else '%RH'
    for split in ['train', 'val', 'test']:
        m = results[target][split]['metrics']
        rows.append({
            'Target': target, 'Split': split, 'Unit': unit,
            'RMSE': round(m['RMSE'], 3), 'MAE': round(m['MAE'], 3),
            'MAPE (%)': round(m['MAPE'], 2), 'R2': round(m['R2'], 4),
        })
metric_df = pd.DataFrame(rows).set_index(['Target','Split'])
display(metric_df)


In [ ]:
# ── Bar chart: RMSE ทุก split ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('RMSE Comparison — Train / Val / Test', fontsize=13, fontweight='bold')

colors = {'train': '#2ecc71', 'val': '#f39c12', 'test': '#e74c3c'}
labels = {'temp': 'Temperature (°C)', 'hum': 'Humidity (%)'}

for ax, target in zip(axes, ['temp', 'hum']):
    splits = ['train', 'val', 'test']
    rmse_vals = [results[target][s]['metrics']['RMSE'] for s in splits]
    bars = ax.bar(splits, rmse_vals, color=[colors[s] for s in splits],
                  alpha=0.8, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, rmse_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(labels[target], fontsize=11)
    ax.set_ylabel('RMSE', fontsize=10)
    ax.set_ylim(0, max(rmse_vals) * 1.25)

plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig2_rmse_splits.png', bbox_inches='tight')
plt.show()


---
## 3. Per-Horizon Metrics (h+1 → h+24)

วัด RMSE และ MAE แยกตามระยะเวลาที่ทำนายล่วงหน้า

In [ ]:
# ── Per-horizon RMSE / MAE ───────────────────────────────────────────────
horizon_rows = []
for target in ['temp', 'hum']:
    yt = results[target]['test']['y_true']
    yp = results[target]['test']['y_pred']
    for h in range(N_HOURS_AHEAD):
        rmse = float(np.sqrt(np.mean((yt[:,h]-yp[:,h])**2)))
        mae  = float(np.mean(np.abs(yt[:,h]-yp[:,h])))
        r2   = float(1 - np.sum((yt[:,h]-yp[:,h])**2) / np.sum((yt[:,h]-yt[:,h].mean())**2))
        horizon_rows.append({'target': target, 'horizon': h+1,
                             'RMSE': rmse, 'MAE': mae, 'R2': r2})

hz_df = pd.DataFrame(horizon_rows)
display(hz_df.pivot_table(index='horizon', columns='target', values='RMSE').round(3))


In [ ]:
# ── Per-horizon RMSE chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Per-Horizon RMSE & MAE on Test Set', fontsize=13, fontweight='bold')

target_style = {'temp': ('#c0392b', '°C'), 'hum': ('#1a5276', '%')}

for ax, metric in zip(axes, ['RMSE', 'MAE']):
    for target, (color, unit) in target_style.items():
        sub = hz_df[hz_df['target']==target].sort_values('horizon')
        ax.plot(sub['horizon'], sub[metric], marker='o', ms=4, lw=2,
                color=color, label=f'{target} ({unit})')
    ax.set_xlabel('Horizon (hours ahead)', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} per Horizon')
    ax.set_xticks(range(1, 25))
    ax.legend(fontsize=10)
    ax.axvline(6,  ls=':', color='#95a5a6', lw=1, alpha=0.7)
    ax.axvline(12, ls=':', color='#95a5a6', lw=1, alpha=0.7)
    ax.axvline(24, ls=':', color='#95a5a6', lw=1, alpha=0.7)
    ax.text(6,  ax.get_ylim()[1]*0.95, '+6h',  ha='center', fontsize=8, color='#7f8c8d')
    ax.text(12, ax.get_ylim()[1]*0.95, '+12h', ha='center', fontsize=8, color='#7f8c8d')

plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig3_horizon_rmse.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Horizon images from training (ถ้ามี) ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, fname in zip(axes, ['lstm_horizon_rmse_temp.png','lstm_horizon_rmse_hum.png']):
    p = MODEL_DIR / fname
    if p.exists():
        ax.imshow(plt.imread(str(p))); ax.axis('off')
        ax.set_title(fname.replace('.png',''), fontsize=10)
    else:
        ax.text(0.5,0.5,'not found',ha='center',transform=ax.transAxes,color='gray')
        ax.axis('off')
plt.suptitle('Horizon RMSE (saved during training run)', fontsize=12)
plt.tight_layout()
plt.show()


---
## 4. Actual vs Predicted — Time Series

เปรียบเทียบค่าจริงกับค่าที่ model ทำนาย บน test set

> แต่ละ prediction window ใช้ข้อมูล 24h ที่ผ่านมาทำนาย 24h ข้างหน้า
> กราฟแสดง h+1 (ทำนาย 1h ล่วงหน้า) และ h+12 (12h)

In [ ]:
# ── Plot actual vs predicted h+1, h+6, h+12, h+24 ────────────────────────
N_SHOW = 7 * 24  # แสดง 7 วัน (~7*24 hourly windows)

for target, color, unit in [
    ('temp', '#c0392b', '°C'),
    ('hum',  '#1a5276', '%'),
]:
    yt = results[target]['test']['y_true']  # (N, 24)
    yp = results[target]['test']['y_pred']
    n  = min(N_SHOW, len(yt))
    idx = np.arange(n)

    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    fig.suptitle(f'{target.upper()} — Actual vs Predicted (last {n} windows of test)',
                 fontsize=13, fontweight='bold')

    for ax, h_minus1 in zip(axes, [0, 5, 11, 23]):
        ax.fill_between(idx, yt[-n:, h_minus1], alpha=0.15, color='#7f8c8d')
        ax.plot(idx, yt[-n:, h_minus1], lw=1.3, color='#2c3e50', label='Actual')
        ax.plot(idx, yp[-n:, h_minus1], lw=1.1, color=color, alpha=0.85,
                label=f'Predicted h+{h_minus1+1}')
        rmse_h = float(np.sqrt(np.mean((yt[-n:, h_minus1]-yp[-n:, h_minus1])**2)))
        ax.set_ylabel(unit, fontsize=10)
        ax.set_title(f'Horizon +{h_minus1+1}h  |  RMSE = {rmse_h:.3f} {unit}',
                     fontsize=10)
        ax.legend(loc='upper right', fontsize=9)

    axes[-1].set_xlabel('Window index (test set)', fontsize=10)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'perf_fig4_actual_pred_{target}.png', bbox_inches='tight')
    plt.show()


---
## 5. Scatter Plot & Residuals Analysis

- Scatter: ถ้า model ดี จุดควรอยู่ใกล้เส้น y=x
- Residuals: ควรกระจายสม่ำเสมอรอบ 0 (ไม่มี bias)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Scatter & Residual Analysis — Test Set', fontsize=13, fontweight='bold')

for row, (target, color, unit) in enumerate([
    ('temp', '#c0392b', '°C'), ('hum', '#1a5276', '%')]):
    yt = results[target]['test']['y_true'].flatten()
    yp = results[target]['test']['y_pred'].flatten()
    res = yt - yp

    # Scatter actual vs predicted
    ax = axes[row, 0]
    ax.scatter(yt, yp, alpha=0.04, s=3, color=color)
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect')
    sl, ic, r, _, _ = stats.linregress(yt, yp)
    xf = np.linspace(lo, hi, 200)
    ax.plot(xf, sl*xf+ic, lw=1.5, color='#e74c3c', label=f'Fit r={r:.3f}')
    ax.set_xlabel(f'Actual ({unit})', fontsize=10)
    ax.set_ylabel(f'Predicted ({unit})', fontsize=10)
    ax.set_title(f'{target.upper()} — Actual vs Predicted')
    ax.legend(fontsize=9)
    ax.text(0.05, 0.9, f'R={r:.3f}', transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='#ffeaa7', alpha=0.8))

    # Residuals vs predicted
    ax = axes[row, 1]
    ax.scatter(yp, res, alpha=0.04, s=3, color=color)
    ax.axhline(0, color='black', lw=1.5, ls='--')
    ax.axhline(res.std(),  color='#e67e22', lw=1, ls=':', label='+1 SD')
    ax.axhline(-res.std(), color='#e67e22', lw=1, ls=':', label='-1 SD')
    ax.set_xlabel(f'Predicted ({unit})', fontsize=10)
    ax.set_ylabel('Residual (Actual - Pred)', fontsize=10)
    ax.set_title(f'{target.upper()} — Residuals')
    ax.legend(fontsize=8)

    # Residuals histogram
    ax = axes[row, 2]
    ax.hist(res, bins=80, color=color, alpha=0.7, edgecolor='white', lw=0.3)
    ax.axvline(0, color='black', lw=1.5, ls='--')
    ax.axvline(res.mean(), color='#e74c3c', lw=1.5,
               label=f'mean={res.mean():.3f}')
    ax.set_xlabel(f'Residual ({unit})', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{target.upper()} — Residual Distribution')
    ax.legend(fontsize=9)
    # overlay normal fit
    mu, sigma = res.mean(), res.std()
    x_norm = np.linspace(res.min(), res.max(), 200)
    pdf = stats.norm.pdf(x_norm, mu, sigma)
    ax2 = ax.twinx()
    ax2.plot(x_norm, pdf, lw=2, color='#2c3e50', alpha=0.7, label='Normal fit')
    ax2.set_ylabel('Density', fontsize=9)
    ax2.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig5_scatter_residuals.png', bbox_inches='tight')
plt.show()


---
## 6. Best / Worst Prediction Windows

หา window ที่ model ทำนายได้ดีที่สุด / แย่ที่สุด (วัดด้วย RMSE per window)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('Best / Worst Prediction Windows (h+1 to h+24)',
             fontsize=13, fontweight='bold')

for row, (target, color, unit) in enumerate([
    ('temp', '#c0392b', '°C'), ('hum', '#1a5276', '%')]):
    yt = results[target]['test']['y_true']
    yp = results[target]['test']['y_pred']

    # per-window RMSE
    win_rmse = np.sqrt(np.mean((yt - yp)**2, axis=1))
    best_idx = np.argsort(win_rmse)[:2]
    worst_idx = np.argsort(win_rmse)[-2:][::-1]
    h_axis = np.arange(1, N_HOURS_AHEAD+1)

    for col, (title, idxs, lcolor) in enumerate([
        ('Best', best_idx,  '#27ae60'),
        ('Worst', worst_idx, '#c0392b'),
    ]):
        for sub_col, win_i in enumerate(idxs):
            ax = axes[row, col*2 + sub_col]
            ax.plot(h_axis, yt[win_i], 'o-', lw=2, color='#2c3e50',
                    ms=4, label='Actual')
            ax.plot(h_axis, yp[win_i], 's--', lw=2, color=lcolor,
                    ms=4, label='Predicted', alpha=0.85)
            ax.fill_between(h_axis, yt[win_i], yp[win_i], alpha=0.12, color=lcolor)
            ax.set_xlabel('Horizon (h)', fontsize=9)
            ax.set_ylabel(unit, fontsize=9)
            ax.set_title(f'{target.upper()} {title} #{sub_col+1}\n'
                         f'RMSE={win_rmse[win_i]:.3f} {unit}', fontsize=9)
            ax.legend(fontsize=8)
            ax.set_xticks([1, 6, 12, 18, 24])

plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig6_best_worst.png', bbox_inches='tight')
plt.show()


---
## 7. Error Heatmap — Hour of Day × Horizon

ตรวจว่า model ทำนายผิดมากช่วงไหนของวัน และที่ horizon ไหน

In [ ]:
# ── สร้าง hour-of-day สำหรับแต่ละ prediction window ────────────────────
# window i ใช้ข้อมูลจนถึง test_df.index[i + LOOKBACK_STEPS - 1]
# ดังนั้น hour of prediction = hour ของ index นั้น
n_windows = results['temp']['test']['y_true'].shape[0]
start_idx  = LOOKBACK_STEPS  # index แรกของ window
pred_hours = test_df.index[start_idx : start_idx + n_windows].hour

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Absolute Error Heatmap — Hour of Prediction vs Horizon',
             fontsize=13, fontweight='bold')

for ax, (target, cmap) in zip(axes, [('temp','YlOrRd'),('hum','Blues')]):
    yt = results[target]['test']['y_true']
    yp = results[target]['test']['y_pred']
    ae = np.abs(yt - yp)  # (N, 24)

    # group by hour_of_day
    heatmap = np.zeros((24, 24))  # hour_of_day x horizon
    for h in range(24):
        mask = (pred_hours == h)
        if mask.sum() > 0:
            heatmap[h, :] = ae[mask].mean(axis=0)

    im = ax.imshow(heatmap, aspect='auto', cmap=cmap, origin='lower')
    plt.colorbar(im, ax=ax, label='Mean Abs Error')
    ax.set_xlabel('Horizon (hours ahead)', fontsize=10)
    ax.set_ylabel('Hour of Day (prediction time)', fontsize=10)
    ax.set_xticks(range(0, 24, 2))
    ax.set_xticklabels([f'+{h+1}h' for h in range(0,24,2)], fontsize=8)
    ax.set_yticks(range(0, 24, 2))
    ax.set_yticklabels([f'{h:02d}:00' for h in range(0,24,2)], fontsize=8)
    ax.set_title(f'{target.upper()} Mean Absolute Error')

plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig7_error_heatmap.png', bbox_inches='tight')
plt.show()


---
## 8. Summary Dashboard

ตารางสรุปผลประเมิน model พร้อมการแปลความหมาย

In [ ]:
print('='*60)
print('  MODEL PERFORMANCE SUMMARY — Test Set')
print('='*60)

for target, unit, good_rmse, good_r2 in [
    ('temp', 'degC', 1.5, 0.7),
    ('hum',  '%RH', 4.0, 0.7),
]:
    m = results[target]['test']['metrics']
    rmse_ok = m['RMSE'] < good_rmse
    r2_ok   = m['R2']   > good_r2
    print(f'\n  {target.upper()} ({unit})')
    print(f'    RMSE : {m["RMSE"]:.3f}  {"OK" if rmse_ok else "NEEDS IMPROVEMENT"}')
    print(f'    MAE  : {m["MAE"]:.3f}')
    print(f'    MAPE : {m["MAPE"]:.2f}%')
    print(f'    R2   : {m["R2"]:.4f}  {"OK" if r2_ok else "LOW — consider retraining"}')

print('\n  Per-Horizon summary:')
print(f'  {"":6} | {"Temp RMSE":>10} | {"Hum RMSE":>10}')
print(f'  {"-"*6}-+-{"-"*10}-+-{"-"*10}')
for h in [0, 5, 11, 23]:
    yt_t = results['temp']['test']['y_true'][:,h]
    yp_t = results['temp']['test']['y_pred'][:,h]
    yt_h = results['hum']['test']['y_true'][:,h]
    yp_h = results['hum']['test']['y_pred'][:,h]
    rt = np.sqrt(np.mean((yt_t-yp_t)**2))
    rh = np.sqrt(np.mean((yt_h-yp_h)**2))
    print(f'  h+{h+1:>2}   | {rt:>10.3f} | {rh:>10.3f}')
print('='*60)


In [ ]:
# ── Final overview figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Performance Summary Dashboard', fontsize=14, fontweight='bold')

# (0,0) Metrics table as heatmap
ax = axes[0, 0]
tab_data = []
tab_rows = []
for target in ['temp', 'hum']:
    for split in ['train','val','test']:
        m = results[target][split]['metrics']
        tab_data.append([m['RMSE'], m['MAE'], m['MAPE'], m['R2']])
        tab_rows.append(f'{target}/{split}')
tab_arr = np.array(tab_data)
im = ax.imshow(tab_arr, cmap='RdYlGn', aspect='auto')
ax.set_xticks([0,1,2,3])
ax.set_xticklabels(['RMSE','MAE','MAPE%','R2'], fontsize=10)
ax.set_yticks(range(len(tab_rows)))
ax.set_yticklabels(tab_rows, fontsize=9)
for i in range(len(tab_rows)):
    for j in range(4):
        ax.text(j, i, f'{tab_arr[i,j]:.2f}', ha='center', va='center',
                fontsize=8, fontweight='bold')
ax.set_title('Metrics Heatmap (train/val/test)')

# (0,1) MAPE by split
ax = axes[0, 1]
x = np.arange(3)
w = 0.35
mape_t = [results['temp'][s]['metrics']['MAPE'] for s in ['train','val','test']]
mape_h = [results['hum'][s]['metrics']['MAPE']  for s in ['train','val','test']]
ax.bar(x-w/2, mape_t, w, color='#c0392b', alpha=0.8, label='Temp')
ax.bar(x+w/2, mape_h, w, color='#1a5276', alpha=0.8, label='Hum')
ax.set_xticks(x)
ax.set_xticklabels(['Train','Val','Test'])
ax.set_ylabel('MAPE (%)')
ax.set_title('MAPE by Split')
ax.legend(fontsize=9)

# (1,0) Per-horizon RMSE both targets
ax = axes[1, 0]
for target, color, unit in [('temp','#c0392b','°C'),('hum','#1a5276','%')]:
    sub = hz_df[hz_df['target']==target].sort_values('horizon')
    ax.plot(sub['horizon'], sub['RMSE'], lw=2, marker='o', ms=3,
            color=color, label=f'{target} ({unit})')
ax.set_xlabel('Horizon (h ahead)')
ax.set_ylabel('RMSE')
ax.set_title('RMSE per Horizon — Test Set')
ax.legend(fontsize=10)
ax.set_xticks(range(1,25))

# (1,1) R2 per horizon
ax = axes[1, 1]
for target, color in [('temp','#c0392b'),('hum','#1a5276')]:
    sub = hz_df[hz_df['target']==target].sort_values('horizon')
    ax.plot(sub['horizon'], sub['R2'], lw=2, marker='o', ms=3,
            color=color, label=target)
ax.axhline(0, ls='--', color='#e74c3c', lw=1, alpha=0.7, label='R2=0 baseline')
ax.axhline(0.7, ls=':', color='#27ae60', lw=1, alpha=0.7, label='R2=0.7 target')
ax.set_xlabel('Horizon (h ahead)')
ax.set_ylabel('R2')
ax.set_title('R2 per Horizon — Test Set')
ax.legend(fontsize=9)
ax.set_xticks(range(1,25))

plt.tight_layout()
plt.savefig(FIG_DIR / 'perf_fig8_dashboard.png', bbox_inches='tight')
plt.show()
print('All figures saved to notebooks/')
